In [2]:
import sys
import os

# 1. Direct Python to the root project folder
sys.path.append(os.path.abspath(os.path.join('..')))

In [5]:
import pandas as pd

df_ground_truth = pd.read_csv("../data/ground_truth.csv")

In [6]:
df_ground_truth.head()

,question,document
0,How do I make an account on your website?,qDNpG85o
1,Where do I click to sign up for a new account?,qDNpG85o
2,What’s the easiest way to register on your site?,qDNpG85o
3,Can you tell me how to create an account online?,qDNpG85o
4,How do I complete the account registration pro...,qDNpG85o


In [7]:
ground_truth = df_ground_truth.to_dict(orient="records")

In [8]:
from scripts.ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

Existing Ecommerce FAQ With Ids loaded Successfully!!
Document Size:79


In [9]:
boost = {"question":3.0}
index.search(
    "How can i sign up my account?", 
    num_results=5, 
    boost_dict=boost
)

[{'question': 'How can I create an account?',
  'answer': "To create an account, click on the 'Sign Up' button on the top right corner of our website and follow the instructions to complete the registration process.",
  'id': 'qDNpG85o'},
 {'question': 'How can I track my order?',
  'answer': "You can track your order by logging into your account and navigating to the 'Order History' section. There, you will find the tracking information for your shipment.",
  'id': 'LOdDd5N3'},
 {'question': 'Can I order without creating an account?',
  'answer': 'Yes, you can place an order as a guest without creating an account. However, creating an account offers benefits such as order tracking and easier future purchases.',
  'id': '7VOpywBa'},
 {'question': 'How can I leave a product review?',
  'answer': "To leave a product review, navigate to the product page on our website and click on the 'Write a Review' button. You can share your feedback and rating based on your experience with the product

In [10]:
def text_search(query):
    boost_dict = {"question": 3.0}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict
    )

In [11]:
q = ground_truth[0]
doc_id = q['document']
results = text_search(q['question'])

In [12]:
for d in results:
    print(f'{d["id"]} == {doc_id}: {d["id"] == doc_id}')

qDNpG85o == qDNpG85o: True
IF0SwlPM == qDNpG85o: False
7VOpywBa == qDNpG85o: False
b4VCHw4X == qDNpG85o: False
LOdDd5N3 == qDNpG85o: False


In [13]:
relevance = []

for d in results:
    relevance.append(int(d["id"] == doc_id))

relevance

[1, 0, 0, 0, 0]

In [14]:
def compute_relevance_text(q):
    doc_id = q["document"]
    results = text_search(query=q["question"])

    relevance = []

    for d in results:
        relevance.append(int(d["id"] == doc_id))

    return relevance

In [15]:
compute_relevance_text(q)

[1, 0, 0, 0, 0]

In [16]:
q = ground_truth[110]

print(q["question"])
compute_relevance_text(q)

Do you have expedited shipping for faster delivery?


[1, 0, 0, 0, 0]

In [17]:
from tqdm.auto import tqdm

def compute_relevance_total_text(ground_truth):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance_text(q)
        relevance_total.append(relevance)

    return relevance_total

In [18]:
relevance = compute_relevance_total_text(ground_truth)

  0%|          | 0/395 [00:00<?, ?it/s]

In [19]:
relevance[:15]

[[1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0]]

In [20]:
def compute_relevance(q,search_function):
    doc_id = q["document"]
    results = search_function(query=q["question"])

    relevance = []

    for d in results:
        relevance.append(int(d["id"] == doc_id))

    return relevance

In [21]:
from tqdm.auto import tqdm

def compute_relevance_total(ground_truth,search_function):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance(q,search_function)
        relevance_total.append(relevance)

    return relevance_total

In [22]:
relevance_total = compute_relevance_total(ground_truth,text_search)

  0%|          | 0/395 [00:00<?, ?it/s]

In [23]:
relevance_total[:15]

[[1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0]]

In [24]:
cnt = 0

for line in relevance_total:
    if 1 in line:
        cnt = cnt + 1

cnt/len(relevance_total)

0.850632911392405

In [25]:
def hit_rate(relevance):
    cnt = 0

    for line in relevance:
        if 1 in line:
            cnt = cnt + 1

    return cnt/len(relevance)

In [26]:
hit_rate(relevance_total)

0.850632911392405

In [27]:
def mrr(relevance):
    total_score = 0.0

    for line in relevance:
        for rank in range(len(line)):
            if line[rank] == 1:
                score = 1 / (rank + 1)
                total_score = total_score + score
                break

    return total_score/len(relevance)

In [28]:
mrr(relevance_total)

0.6845569620253166

In [29]:
def evaluate(ground_truth, search_function):
    relevance_total = compute_relevance_total(ground_truth,search_function)

    return {
        "hit_rate": hit_rate(relevance_total),
        "mrr": mrr(relevance_total)
    }

In [30]:
def text_search_V2(query):
    boost_dict = {"question": 0.5}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict
    )

In [31]:
evaluate(ground_truth, text_search_V2)

  0%|          | 0/395 [00:00<?, ?it/s]

{'hit_rate': 0.9037974683544304, 'mrr': 0.7432067510548523}

In [32]:
def search_boost(query, question_boost):
    boost_dict = {"question": question_boost}
    
    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict
    )

In [33]:
for boost in [0.5, 1.0, 3.0, 5.0, 10.0]:
    result = evaluate(
        ground_truth,
        lambda query, boost=boost: search_boost(query,boost)
    )
    print(f"boost={boost}: {result}")

  0%|          | 0/395 [00:00<?, ?it/s]

boost=0.5: {'hit_rate': 0.9037974683544304, 'mrr': 0.7432067510548523}


  0%|          | 0/395 [00:00<?, ?it/s]

boost=1.0: {'hit_rate': 0.8886075949367088, 'mrr': 0.7370042194092827}


  0%|          | 0/395 [00:00<?, ?it/s]

boost=3.0: {'hit_rate': 0.850632911392405, 'mrr': 0.6845569620253166}


  0%|          | 0/395 [00:00<?, ?it/s]

boost=5.0: {'hit_rate': 0.830379746835443, 'mrr': 0.6680590717299579}


  0%|          | 0/395 [00:00<?, ?it/s]

boost=10.0: {'hit_rate': 0.810126582278481, 'mrr': 0.6469620253164559}


In [34]:
def search_boosts(query, question_boost, answer_boost):
    boost_dict = {
        "question": question_boost,
        "answer": answer_boost
    }

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
    )

In [35]:
results = []

for question_boost in [1.0, 2.0, 5.0]:
    for answer_boost in [1.0, 2.0, 4.0, 10.0]:
        print(
            f"Evaluating question_boost={question_boost},"
            f" answer_boost={answer_boost}..."
        )
        result = evaluate(
            ground_truth,
            lambda query, question_boost=question_boost, answer_boost=answer_boost: search_boosts(
                query,
                question_boost,
                answer_boost
            )
        )

        results.append({
            "question": question_boost,
            "answer": answer_boost,
            "hit_rate": result["hit_rate"],
            "mrr": result["mrr"],
        })

Evaluating question_boost=1.0, answer_boost=1.0...


  0%|          | 0/395 [00:00<?, ?it/s]

Evaluating question_boost=1.0, answer_boost=2.0...


  0%|          | 0/395 [00:00<?, ?it/s]

Evaluating question_boost=1.0, answer_boost=4.0...


...

  0%|          | 0/395 [00:00<?, ?it/s]

Evaluating question_boost=5.0, answer_boost=4.0...


  0%|          | 0/395 [00:00<?, ?it/s]

Evaluating question_boost=5.0, answer_boost=10.0...


  0%|          | 0/395 [00:00<?, ?it/s]

In [36]:
df_results = pd.DataFrame(results)
df_results.sort_values("mrr", ascending=False).head(10)

,question,answer,hit_rate,mrr
1,1.0,2.0,0.903797,0.743207
6,2.0,4.0,0.903797,0.743207
11,5.0,10.0,0.903797,0.743207
0,1.0,1.0,0.888608,0.737004
5,2.0,2.0,0.888608,0.737004
10,5.0,4.0,0.886076,0.726582
2,1.0,4.0,0.873418,0.719873
7,2.0,10.0,0.865823,0.714768
4,2.0,1.0,0.860759,0.702447
3,1.0,10.0,0.853165,0.698650
